# AI/ML Final Project
## Retrieval-Augmented Generation (RAG) System using LangChain and FAISS

**Name:** Mansi Pareek
**Internship:** AIML Crash Course[code trade India Pvt. Ltd.]

In [35]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
folder_path = "RAG_Project_Starter_Kit/Data"

documents = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        loader = TextLoader(file_path, encoding="utf-8")
        documents.extend(loader.load())

print("Number of documents loaded:", len(documents))

Number of documents loaded: 10


## Step 2: Split Documents into Chunks

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks created:", len(chunks))

Number of chunks created: 394


## Step 3: Create Embeddings

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully!")

C:\Users\hp\AppData\Local\Temp\ipykernel_10352\2998448268.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\hp\OneDrive\Desktop\aiml_final_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variab

Embeddings model loaded successfully!


## Step 4: Create the FAISS Vector Database

In [14]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector database created successfully!")
print("Total vectors stored:", vectorstore.index.ntotal)

FAISS vector database created successfully!
Total vectors stored: 394


## Step 5: Create the Retriever


In [15]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")


Retriever created successfully!


## Step 6: Configure Google Gemini

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "enter your API key"
print("API Key configured successfully!")

API Key configured successfully!


## Step 7: Create the RAG Chain (Retriever + LLM + Prompt)

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Create Prompt Template
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Use ONLY the following context to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and concisely.
""")

# Step 2: Output Parser
output_parser = StrOutputParser()

print("Prompt and output parser created successfully!")

Prompt and output parser created successfully!


## Step 8: Build the RAG Chain (Retriever + Prompt + LLM)

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini model initialized successfully!")


Gemini model initialized successfully!


In [22]:
from langchain_core.runnables import RunnablePassthrough

# Function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Correct RAG Chain (compatible with your setup)
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | output_parser
)

print("RAG Chain created successfully!")

RAG Chain created successfully!


In [23]:
rag_chain.invoke("What is in the employee handbook?")

'The context states that the "AURAHEALTH NEXUS - EMPLOYEE HANDBOOK 2026" is a "CONFIDENTIAL INTERNAL DOCUMENT" and is "VERSION 4.2". It does not specify what content is within the handbook.'

In [24]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a precise AI assistant for an internal company document system.

Use ONLY the given context to answer the question.
If the answer is not found in the context, clearly say:
"Answer not found in the provided documents."

Context:
{context}

Question:
{question}

Provide a clear, accurate, and direct answer.
""")

print("Improved prompt loaded successfully!")

Improved prompt loaded successfully!


In [25]:
rag_chain.invoke("What is AURAHEALTH NEXUS?")

'AuraHealth Nexus is an organization founded on the principle that the integration of advanced biotechnology and artificial intelligence is the key to unlocking the next stage of human evolution. It has state-of-the-art facilities across the globe dedicated to pushing the boundaries of medical science and is positioned as the undisputed leader in next-generation healthcare solutions.'

In [26]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

## Step 9: Testing the RAG System

In this step, we will test the Retrieval-Augmented Generation (RAG) pipeline using different questions to evaluate how well the system retrieves relevant information from the documents and generates accurate responses using the Gemini LLM.

The model answers are based only on the context retrieved from the FAISS vector database.

In [27]:
rag_chain.invoke("What is AuraHealth Nexus?")

'AuraHealth Nexus is a company founded on the principle that the integration of advanced biotechnology and artificial intelligence is the key to unlocking the next stage of human evolution. It operates state-of-the-art facilities across the globe, dedicated to pushing the boundaries of medical science, and is positioned as the undisputed leader in next-generation healthcare solutions.'

In [28]:
rag_chain.invoke("What does the employee handbook contain?")

'The employee handbook is the "AURAHEALTH NEXUS - EMPLOYEE HANDBOOK 2026, VERSION 4.2" and is a "CONFIDENTIAL INTERNAL DOCUMENT."'

In [29]:
rag_chain.invoke("What are the security protocols?")

'The security protocols are classified as Level 4 Restricted and are described as the most stringent for Sector 7. They include a Biocontainment Breach Protocol and AI Systems and Interaction Protocols.'

## Observation

The RAG system successfully retrieves relevant document chunks from the FAISS vector database and generates context-aware responses using the Gemini LLM. The answers are strictly based on the provided documents.

## Step 10: Final Evaluation Testing
We are testing the RAG system using multiple queries to evaluate retrieval accuracy and response quality.

In [31]:
rag_chain.invoke("What is CryoStasis Recovery Procedure?")


'The provided context describes a specific part of the CryoStasis Recovery Procedure, namely the "POST-THAW NEUROLOGICAL ASSESSMENT" and the intervention required if a patient scores below 75 on the Vellox Cognitive Battery. However, it does not provide a general definition of what the overall CryoStasis Recovery Procedure is.'

In [32]:
rag_chain.invoke("What is NeuroCrystal Syndrome?")

'NeuroCrystal Syndrome is a medical syndrome where crystalline formations interfere significantly with the peripheral nervous system, causing severe neuropathy and localized paralysis.'

In [33]:
rag_chain.invoke("What is OmniHeal project?")

'The OmniHeal project is an initiative making groundbreaking progress in nanite-assisted surgery, which promises to reduce recovery times by up to 80%.'

## Conclusion

The Retrieval-Augmented Generation (RAG) system was successfully implemented using LangChain, FAISS, and Google Gemini. The system effectively retrieves relevant document chunks and generates accurate context-based responses.

This project demonstrates how AI can be used for intelligent document understanding and question answering systems.